In [1]:
import numpy as np
import pandas as pd

In [2]:
# Load in the dataset
df = pd.read_csv('../../datasets/heart.csv')

X = df.drop(columns=['target']).to_numpy()
y = df['target'].to_numpy().reshape((303,1)) # Force the 1 dimension

# Denote the categorical features in the csv file
categorical_features_idx = [1, 2, 5, 6, 8, 10, 11]

In [3]:
# Split into train and test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, shuffle=True)

In [4]:
# Calculate the gini impurity of a node
def gini(examples: np.ndarray):
    """
    Given an examples array holding the classes of each example,
    calculate the gini impurity of the current node.
    """
    N = len(examples)

    # Get all unique classes
    _, counts = np.unique(examples, return_counts=True)

    # Find proportions and apply gini formula
    probabilities = counts / N

    return 1 - np.sum(probabilities)

def weighted_gini(y, mask):
    N = len(y)

    # Get the positive/negative (left/right) examples
    y_left = y[mask]
    y_right = y[~mask]

    # Calculate impurities
    gini_left = gini(y_left)
    gini_right = gini(y_right)

    # Calculate weighted average
    weighted_avg = (len(y_left) / N) * gini_left + (len(y_right) / N) * gini_right

    return weighted_avg

In [5]:
# Node for decision tree
class DTNode:
    def __init__(self, feature=None, threshold=None, mode="numerical", pred=None, left=None, right=None):
        # Store split features
        self.feature = feature
        self.threshold = threshold
        self.mode = mode
        
        self.pred = pred

        # Matches split condition, go left
        self.left: DTNode | None = left

        # Fails split condition, go right
        self.right: DTNode | None = right

    def is_leaf(self):
        return self.left == self.right == None

    # Go down the nodes until find a leaf node--then predict
    def predict(self, x):
        if self.is_leaf():
            return self.pred
        
        if self.mode == "numerical":
            return self.left.predict(x) if x[self.feature] < self.threshold else self.right.predict(x)
        else:
            return self.left.predict(x) if x[self.feature] == self.threshold else self.right.predict(x)

In [6]:
# TODO: Make subsampling of numerical features, because when you have 10,000 midpoints, it gets kinda rough

class DecisionTreeClassifier:
    def __init__(self, max_depth=10, categorical_features=[]):
        self.max_depth = max_depth

        # Initialize the binary tree
        self.root = None

        # Denote certain columns as categorical
        self.categorical_features = categorical_features

    def find_best_split(self, X: np.ndarray, y: np.ndarray):
        # Keep track of best split
        best_gini = np.inf
        best_split_info = None # In the form (feature_index, split_threshold, split_type)

        # Traverse through each feature
        for i, col in enumerate(X.T):
            unique = np.unique(col) # Get all unique elements
            
            # Don't try if there's only one unique element
            if len(unique) == 1:
                continue

            # Categorical type
            is_categorical = (i in self.categorical_features) or isinstance(col[0], str)
            if is_categorical:

                for category in unique:
                    mask = (col == category)

                    # Get weighted avg
                    weighted_avg = weighted_gini(y, mask)

                    # Update best category
                    if weighted_avg < best_gini:
                        best_gini = weighted_avg
                        best_split_info = (i, category, "categorical")
            else:
                # Sort the numerical array unique values
                sorted_vals = np.sort(unique)

                # Calculate pairs of medians:
                medians = (sorted_vals[:-1] + sorted_vals[1:]) / 2

                # Find best split among medians
                for median in medians:
                    mask = (col < median)

                    # Get weighted avg
                    weighted_avg = weighted_gini(y, mask)

                    # Update best category
                    if weighted_avg < best_gini:
                        best_gini = weighted_avg
                        best_split_info = (i, median, "numerical")

        return best_split_info

    def build_tree(self, X: np.ndarray, y: np.ndarray, depth = 1):
        unique, counts = np.unique(y, return_counts=True)
        majority_class = unique[np.argmax(counts)]

        # Don't go past max_depth
        if depth > self.max_depth:
            return DTNode(pred=majority_class)
        
        # If only one category in y, don't go further
        if len(unique) == 1:
            return DTNode(pred=majority_class)
        
        # Find the best split at the current state
        split = self.find_best_split(X, y)

        # If no best split, make leaf
        if split == None:
            return DTNode(pred=majority_class)

        # Create a new node
        feature_idx, threshold, split_type = split
        node = DTNode(feature_idx, threshold, split_type)

        # Find the best splits for left and right node
        if split_type == "categorical":
            mask = (X[:, feature_idx] == threshold)
        else:
            mask = (X[:, feature_idx] < threshold)
        
        # Get examples and features for left and right node
        X_left, y_left = X[mask], y[mask]
        X_right, y_right = X[~mask], y[~mask]

        # Recursively add nodes
        node.left = self.build_tree(X_left, y_left, depth + 1)
        node.right = self.build_tree(X_right, y_right, depth + 1)

        # Return the final node
        return node
    
    def fit(self, X, y):
        """
        Builds optimal decision tree based on data
        """
        self.root = self.build_tree(X, y)

    def predict(self, X):
        """
        Predits a group of examples
        """
        return np.array([self.root.predict(x) for x in X])

In [34]:
model = DecisionTreeClassifier(max_depth=20, categorical_features=categorical_features_idx)
model.fit(X_train, y_train)

In [35]:
# Evaluate the model
y_pred = model.predict(X_test)
flattened_y_test = np.squeeze(y_test, axis=1)

correct = (flattened_y_test == y_pred).astype(int)

accuracy = np.sum(correct) / len(correct)

print(f'Accuracy: {accuracy:.3f}')

Accuracy: 0.652
